[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/relativity/closed_universe/waves.ipynb)

# Gravitational Waves of a Closed Universe

This example computes the gravitational waves of a closed universe, the round three-sphere, on the 600-cell's regular tetrahedra, and watches them from inside, through small crystals of atoms on springs. Regge calculus holds a curved space in the lengths of a mesh's edges: each tetrahedron is a piece of the space with those lengths, and the curvature gathers at the edges, as the deficit by which the angles of the tetrahedra round an edge fall short of a whole turn. The tetrahedra here are spherical, each a piece of the round sphere rather than a flat simplex, as [Bahr and Dittrich (2009)](https://doi.org/10.1103/PhysRevD.80.124030) build Regge calculus from simplices of constant curvature: the round sphere is then exactly at rest, with no deficit at any edge of any mesh, where flat tetrahedra would leave the 600-cell a deficit of about seven degrees at every edge, and every deficit that remains is a wave's. A gravitational wave changes the lengths, and the deficits it causes pull them back. The slowest wave of the closed universe, its squash, stretches the whole sphere along one direction and squeezes it along another, and swings back.

Points of the sphere are modelled as antivectors, and a tetrahedron's ramps, the planes through its faces scaled to read one at the opposite corners, read a point's barycentric weights, `ramps & point`. With the squared lengths, the ramps' products give the metric of the cone over the tetrahedron from the sphere's centre, a map from points to planes, and through its inverse two ramps pair to the cosine of the angle between their faces: read so, each tetrahedron is a piece of the round sphere, and the round sphere's own chords leave no deficit on any mesh. The lengths' kinetic term is the inertia of space, a map from the rates of the squared lengths to their momenta, amounts, so that the kinetic energy is `masses(rates) & rates / 2`. The waves are the changes no motion of the vertices makes, kept so by one solve over the vertices; stepped through time they make a smooth field of strain, read in the isoclinic frames, at every place a map from a step of the origin to the plane that reads its stretch, `strain(step) & step`, through which the crystals' atoms and the light roll.

In the notation of Regge calculus, the action of a three-space of spherical tetrahedra is $S = \sum_e \ell_e\,\epsilon_e + 2V$ over its edges' arc lengths, their deficit angles and its volume, which by Schläfli's identity for spherical tetrahedra, $\sum_e \ell_e\,\delta\theta_e = 2\,\delta V$ in each, changes as $\delta S = \sum_e \epsilon_e\,\delta\ell_e$, and the linear waves of the squared lengths $q$ solve $M\ddot q = -K q$, with $K$ the Hessian of the potential and $M$ the discrete DeWitt supermetric, restricted to the transverse traceless changes: those $M$-orthogonal to $D u$ for every vertex move $u$, found from the normal equations $D^{\mathsf T} M D\,u = D^{\mathsf T} M\,\delta q$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from functools import reduce
from operator import and_

import numpy as np
from numga import NumpyContext, stack
from numga.sparse import spdiag

from examples.util.animation import show_animation
from examples.util.topology import Topology
from examples.relativity.closed_universe import core, render, scenarios
from examples.relativity.closed_universe.core import ga

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                     # Scalar+
# What is integrated over a volume: the action, and the momenta of the squared lengths.
Amount = ga.gatype.antiscalar()                                                 # Antiscalar-
# Planes: the ramps across the tetrahedra, and those through the origin, the polar planes of its steps.
Plane = ga.gatype.vector()                                                      # Vector+
Through = Plane.derive.within("x y z")                                          # Vector+
# Points of the sphere and the origin's steps; the motors that carry them, and the rates that move and turn them.
Point = ga.gatype.antivector()                                                  # Antivector-
Step = Point.derive.containing("w")                                             # Antivector-
Motor = ga.gatype.rotor()                                                       # Rotor+
Rate = ga.gatype.bivector()                                                     # Bivector+
# A strain, the plane that reads each step's stretch; a turning, the rate a step adds to a frame's turn; a
# stretching, the steps a map of them gives.
Strain = ga.gatype((Through, Step))                                             # Vector+ <- Antivector-
Turning = ga.gatype((Rate, Step))                                               # Bivector+ <- Antivector-
Stretching = ga.gatype((Step, Step))                                            # Antivector- <- Antivector-
Translating = ga.gatype((Step, Through))                                        # Antivector- <- Vector+

# The origin's equator, the plane `w`, and the origin, its pole.
equator = mv.w                                                                  # [] Plane+
origin = equator.dual()                                                         # [] Point-
# The step the origin takes under the motion across a plane through it, a step by its value, cast so; its inverse
# gives each step its polar plane, the round sphere's metric on the origin's steps as a correlation, and the meet of
# the equator with that plane is the rate that moves the origin along the step.
translating = origin.commutator(equator ^ Through).cast(Translating)            # [] Point- <- Plane+
polar = translating.inverse()                                                   # [] Plane+ <- Point-
moving = equator ^ polar(Step)                                                  # [] Rate+ <- Point-

## Isoclinic frames

In four dimensions the product of a rate with the extrinsic pseudoscalar `mv.xyzw` is the rate in the plane square to it, and as the pseudoscalar squares to one, it splits every rate into a half it keeps and a half it flips. Each half turns alike in two square planes, by the same angle in both, so that it moves every point of the sphere the same distance along a great circle. The quotient `place / origin` is the turn that carries the origin to a place in the plane of the great circle between them; its half that the pseudoscalar keeps, with one in the other half, still carries the origin there, turning alike in that plane and in the plane square to it: that motor is the isoclinic frame at the place. Carried along a step, the frames roll by `moving(step) + turning(step)`, where `turning`, the moving rate times the pseudoscalar, turns them in the plane square to it, and in them a heading keeps still along its great circle. The pseudoscalar is what picks out these turns, and with them a frame at every place that varies smoothly over the whole sphere.

In [ ]:
# The extrinsic pseudoscalar, whose product with a rate is the rate in the plane square to it; the rate that turns a
# frame in the plane square to each step's moving rate.
pseudoscalar = mv.xyzw                                                          # [] Antiscalar+
turning = moving * pseudoscalar                                                 # [] Rate+ <- Point-


def isoclinic(places: Point) -> Motor:
    """The isoclinic frames at places on the sphere, `Motor+`: the motors that carry the origin to each by turning
    alike in the plane of the great circle between them and in the plane square to it."""
    # The half of the quotient of the place by the origin that the pseudoscalar keeps, with one in the other half.
    return ((1 + pseudoscalar) * (places.normalized() / origin) + (1 - pseudoscalar)) / 2

## The 600-cell at rest

The 600-cell's 120 vertices lie on the sphere, and its edges' squared lengths at rest are those of the round sphere's chords between them. Read as a piece of the round sphere, a tetrahedron is the cone over it from the sphere's centre, whose metric the squared lengths fix; through its inverse the faces at each edge meet at their angles, and a whole turn less those angles round the edge is the curvature gathered there. By Schläfli's identity a spherical tetrahedron's volume changes by half its arcs times the changes of its angles, so with twice the volume in the action the deficits alone drive the lengths: the action is an integral over the volume, so its slope along each squared length is an amount.

In [ ]:
cell_mesh, cell_vertices = scenarios.six_hundred_cell()
chords = cell_mesh.d[0] * cell_vertices                                         # Point-[edges]
rest = chords.dot(chords)                                                       # Scalar+[edges]
corners = cell_mesh.incidence[3, 0] * cell_vertices                             # [corners] Point-[cells]
# Each corner's ramp: the plane through the face across from it, scaled to read one at the corner.
faces = reduce(and_, corners[cell_mesh.corners[3, 2]])                          # [corners] Plane-[cells]
ramps = faces / (faces & corners)                                               # [corners] Plane+[cells]
# The hyperplane through the corners, one on each, and each pair of ramps' symmetric product, which reads the squared
# chord between their corners: weighted by the squared lengths, the cone's metric from the centre.
radial = ramps.sum(axis=0)                                                      # Plane+[cells]
tails, heads = cell_mesh.corners[3, 1]
cone_basis = -(ramps[tails] * (ramps[heads] & Point) + ramps[heads] * (ramps[tails] & Point)) / 2   # [cell_edges] (Plane+ <- Point-)[cells]
cone_map = (spdiag(cone_basis) * cell_mesh.incidence[3, 1]).sum(axis=0)         # (Plane+ <- Point-)[cells <- edges]
# The corners off each edge, at the faces that meet there.
off_tails, off_heads = cell_mesh.complements[3, 1]


def deficits(squared_lengths: Scalar) -> Scalar:
    """The deficit angle at each edge, `Scalar+[edges]`, of the tetrahedra read as pieces of the round sphere with
    these squared lengths: a whole turn less the angles between the faces round the edge."""
    cone = radial * (radial & Point) + cone_map * squared_lengths               # (Plane+ <- Point-)[cells]
    pairings = ramps[:, None] & cone.batch().inverse().field()(ramps[None, :])   # [corners, corners] Scalar+[cells]
    off_edges = pairings[off_tails, off_heads]                                  # [cell_edges] Scalar+[cells]
    sides = pairings[off_tails, off_tails] * pairings[off_heads, off_heads]     # [cell_edges] Scalar+[cells]
    angles = (-off_edges / sides.square_root()).arccos()                        # [cell_edges] Scalar+[cells]
    return 2 * np.pi - (~cell_mesh.incidence[3, 1] * angles).sum(axis=-1)


def slopes(squared_lengths: Scalar) -> Amount:
    """The action's slope along each squared length, `Amount-[edges]`: the deficit, times how the arc changes with the
    squared chord, a half over the chord and the cosine of half the arc."""
    return (-deficits(squared_lengths) / (squared_lengths * (1 - squared_lengths / 4)).square_root()).dual()

## The inertia of space and the waves

The lengths' motion has relativity's kinetic term, each tetrahedron taken flat: integrated over the volume, it is the inertia of space, `masses`, from the rates of the squared lengths to their momenta, amounts. The stiffness is the change of the action's slopes with the lengths. At rest every deficit vanishes, so only the angles change, and each angle changes through the inverse of the cone's metric, whose change with an edge's squared length is minus the inverse, times the edge's product of ramps, times the inverse: a product of the faces' own pairings, written out with no nudging. A change of the squared lengths that a move of the vertices makes is no wave; the waves are what is left once the nearest such change by the inertia is taken away, the inertia pulled back to the vertex moves by their adjugate, `moves.adjugate()(masses(moves))`.

In [ ]:
# The flat tetrahedra: the corner gradients, their products, and each one's metric and volume from the squared lengths.
gradients = ramps - radial / 4                                                  # [corners] Plane+[cells]
flat_basis = -(gradients[tails] * (gradients[heads] & Point) + gradients[heads] * (gradients[tails] & Point)) / 2   # [cell_edges] (Plane+ <- Point-)[cells]
flat_map = (spdiag(flat_basis) * cell_mesh.incidence[3, 1]).sum(axis=0)         # (Plane+ <- Point-)[cells <- edges]
flat = radial * (radial & Point) + flat_map * rest                              # (Plane+ <- Point-)[cells]
spans = corners[1:] - corners[0]                                                # [spans] Point-[cells]
volumes = ((flat(spans[0]) ^ flat(spans[1]) ^ flat(spans[2]) ^ radial) & (corners[0] & corners[1] & corners[2] & corners[3])).abs().square_root() / 6   # Scalar+[cells]
pairings = gradients[:, None] & flat.batch().inverse().field()(gradients[None, :])   # [corners, corners] Scalar+[cells]
# Relativity's kinetic term between the rates of two edges' squared lengths, assembled over the tetrahedra: amounts.
local = volumes / 4 * (pairings[tails[:, None], tails[None, :]] * pairings[heads[:, None], heads[None, :]]
                       + pairings[tails[:, None], heads[None, :]] * pairings[heads[:, None], tails[None, :]])   # [cell_edges, cell_edges] Scalar+[cells]
edges = cell_mesh.incidence[3, 1]                                               # [cell_edges] numbers[cells <- edges]
masses = (~edges[:, None] * spdiag(local * Scalar.dual()) * edges[None, :]).sum(axis=0).sum(axis=0)   # (Amount- <- Scalar+)[edges <- edges]
# The stiffness, the change of the slopes with the squared lengths at rest: every deficit vanishes there, so only the
# angles change, each through the inverse of the cone's metric, whose change with an edge's squared length is minus the
# inverse, times that edge's product of ramps, times the inverse, a product of the faces' pairings.
cone = radial * (radial & Point) + cone_map * rest                              # (Plane+ <- Point-)[cells]
cone_pairings = ramps[:, None] & cone.batch().inverse().field()(ramps[None, :])   # [corners, corners] Scalar+[cells]
# The corners off each angle's edge, down the rows, and the ends of each squared length's edge, across.
first_off, second_off = off_tails[:, None], off_heads[:, None]
tail_ends, head_ends = tails[None, :], heads[None, :]
# How an angle's two faces pair with each other, and each with itself, change with each squared length.
between = (cone_pairings[first_off, tail_ends] * cone_pairings[second_off, head_ends]
           + cone_pairings[first_off, head_ends] * cone_pairings[second_off, tail_ends]) / 2   # [cell_edges, cell_edges] Scalar+[cells]
own = (cone_pairings[first_off, tail_ends] * cone_pairings[first_off, head_ends] / cone_pairings[first_off, first_off]
       + cone_pairings[second_off, tail_ends] * cone_pairings[second_off, head_ends] / cone_pairings[second_off, second_off])
# The angle's change, through the arccos of the faces' pairing over the root of their own.
facing = cone_pairings[first_off, second_off]                                   # [cell_edges, 1] Scalar+[cells]
sines = (cone_pairings[first_off, first_off] * cone_pairings[second_off, second_off] - facing.squared()).square_root()
angle_changes = (between - facing * own / 2) / sines                            # [cell_edges, cell_edges] Scalar+[cells]
# A slope is minus the deficit over the root of the arc's change with the squared chord, so its change is the angles'
# changes summed round the edge over that root; integrated over the volume, amounts.
roots = edges * (rest * (1 - rest / 4)).square_root()                          # [cell_edges] Scalar+[cells]
stiffness = (~edges[:, None] * spdiag(angle_changes / roots[:, None] * Scalar.dual()) * edges[None, :]).sum(axis=0).sum(axis=0)   # (Amount- <- Scalar+)[edges <- edges]
moves = spdiag(2 * chords.dot(Point)) * cell_mesh.d[0]                          # (Scalar+ <- Point-)[edges <- vertices]
# The moves of the whole sphere change no length: this share of the largest move's own square, in the isotropic form
# on each vertex's moves, the algebra's own dual, settles them at nothing.
settling = 1e-10
reach = moves.adjugate()(masses(moves))                                         # (Plane+ <- Point-)[vertices <- vertices]
reach = reach + settling * core.largest(reach) * Point.dual()


def onto_waves(changes: Scalar) -> Scalar:
    """Changes of the squared lengths kept to the waves, `Scalar+[edges]`: less the change the vertex moves make that is
    nearest them by the inertia."""
    return changes - moves(reach.solve(moves.adjugate()(masses(changes))))

## The squash

Read in the isoclinic frames, an edge runs along a step of the origin, its round step, and the squash lengthens the edges along `x` and shortens those along `y`, the same everywhere in the frames. Kept to the waves, its frequency squared, the force of its stiffness over its momentum, is a fiftieth short of the continuum's eight, on edges a fifth of a half turn long.

In [ ]:
def round_steps(graph: Topology, places: Point) -> Point:
    """Each edge's round step, `Point-[edges]`, a step of the origin: its head carried back by its tail's isoclinic
    frame, less the origin it is beside."""
    tails, heads = graph.incidence[1, 0] * places                               # Point-[edges], Point-[edges]
    return (isoclinic(tails).inverse() >> heads).cast(Step)


steps = round_steps(cell_mesh, cell_vertices)                                   # Point-[edges]
# The squash's change of the squared lengths, per unit swing of space: twice, as a squared length changes twice as
# fast as the length.
squash = 2 * rest * ((mv.x & steps).squared() - (mv.y & steps).squared()) / (polar(steps) & steps)   # Scalar+[edges]
pattern = onto_waves(squash)                                                    # Scalar+[edges]
squash_squared = float(((stiffness(pattern) & pattern).sites.sum() / (masses(pattern) & pattern).sites.sum()).to_array())
print(f"the squash's frequency squared: {squash_squared:.3f}")

## Stepping through time

The waves step by leapfrog, kept to the waves at every kick. Started from the squash at its swing long before the first frame, so that the crystals settle into the swing and the light that reaches the eye has set out, they ring through one swing more, every step kept.

In [ ]:
# The swing of space, the steps to a swing, and the swings the crystals settle over before the frames.
wave_swing = 0.12
n_steps_per_swing = 40
n_settling_swings = 8

In [ ]:
def waves_stepped(strains: Scalar, rates: Scalar, step: float) -> Iterator[tuple[Scalar, Scalar]]:
    """The squared lengths' changes and their rates, `Scalar+[edges]` each, at every step of a leapfrog under the
    stiffness and the inertia, kept to the waves: one solve of the restoring pull a step, the rates kept half a step
    off the changes and given at the changes' time, the mean of the half steps either side."""
    # The stiffness's pull through the inertia, kept to the waves; the rates half a step before the changes.
    restoring = onto_waves(masses.solve(stiffness(strains)))                    # Scalar+[edges]
    behind = rates + (step / 2) * restoring                                     # Scalar+[edges]
    while True:
        ahead = behind - step * restoring                                       # Scalar+[edges]
        yield strains, (behind + ahead) / 2
        strains, behind = strains + step * ahead, ahead
        restoring = onto_waves(masses.solve(stiffness(strains)))


frequency = np.sqrt(squash_squared)
period = 2 * np.pi / frequency
level_step = period / n_steps_per_swing
# From the settling swings and half a turn more, the light's time across the sphere, to the end of the frames' swing.
first_time = -(n_settling_swings + 1) * period - np.pi
n_levels = int(np.ceil((period - first_time) / level_step)) + 2
strains = wave_swing * np.cos(frequency * first_time) * pattern                 # Scalar+[edges]
rates = -wave_swing * frequency * np.sin(frequency * first_time) * pattern      # Scalar+[edges]
levels = [state for _, state in zip(range(n_levels), waves_stepped(strains, rates, level_step))]

## A smooth field of strain

A tetrahedron's strain is the change of its cone metric read in the isoclinic frame at its middle, a map from a step of the origin to the plane that reads its stretch. Read in the frames, the strains of different tetrahedra are maps on the same steps of the origin, which is what lets them be averaged and blended at all. Averaged at the vertices, blended across each tetrahedron by a place's barycentric weights and between the stored moments by their nearness, the strains make a smooth field, and where the strain changes across a tetrahedron, half its curl turns the frames. Through the field the frames roll along a step at a rate the strain bends away from the round sphere's, the connection, which the atoms and the light roll along; the squeeze, half the strain's rate read back as a step, is how fast the slices of space stretch under them. This cell takes several seconds, most of it reading the strain at every stored moment.

In [ ]:
# The isoclinic frame at each tetrahedron's middle, and how many tetrahedra share each vertex.
cell_frames = isoclinic(cell_mesh.incidence[3, 0].mean(axis=0) * cell_vertices)   # Motor+[cells]
counts = (~cell_mesh.incidence[3, 0] * mv.scalar(np.ones((cell_mesh.n_elements[3], 1))).field()).sum(axis=0)   # Scalar+[vertices]


def strain_field(changes: Scalar) -> tuple[Strain, Turning]:
    """The strain of a change of the squared lengths, `Scalar+[edges]`, at each vertex, `Strain[vertices]`: the
    average of its tetrahedra's strains, each the change of the cone's metric read on the steps the frame at the
    tetrahedron's middle carries out, and carried back; and each tetrahedron's turn, half the curl of the strain
    across it, `Turning[cells]`."""
    frame_steps = cell_frames >> Step                                           # (Point- <- Point-)[cells]
    # What reads nothing on the steps, along the origin itself, cast away.
    strains = (cell_frames.inverse() >> (cone_map * changes)(frame_steps)).cast(Strain)   # (Plane+ <- Point-)[cells]
    at_vertices = (~cell_mesh.incidence[3, 0] * strains).sum(axis=0) / counts   # (Plane+ <- Point-)[vertices]
    # Each corner's ramp read on the carried steps, and the strain at each corner: their wedge, summed over the
    # corners, is the strain's change across the tetrahedron turned into the plane it turns in.
    corner_planes = (cell_frames.inverse() >> ramps).cast(Through)              # [corners] Plane+[cells]
    corner_strains = cell_mesh.incidence[3, 0] * at_vertices                    # [corners] (Plane+ <- Point-)[cells]
    return at_vertices, (corner_planes ^ corner_strains).sum(axis=0) / 2


# The strain at the vertices and each tetrahedron's turn at every stored moment, the strain's rate, and its trace.
level_fields = [strain_field(changes) for changes, _ in levels]
level_strains = stack([strain.batch() for strain, _ in level_fields])           # [levels, vertices] Plane+ <- Point-
level_turns = stack([turn.batch() for _, turn in level_fields])                 # [levels, cells] Rate+ <- Point-
level_strain_rates = stack([strain_field(rates)[0].batch() for _, rates in levels])   # [levels, vertices] Plane+ <- Point-
level_traces = translating(level_strains).trace()                               # [levels, vertices] Scalar+

In [ ]:
def connection(cells: np.ndarray, places: Point, time: float) -> tuple[Turning, Stretching]:
    """The connection at the places `Point-[sites]` in the tetrahedra `cells` `[sites]`, at a time: the rate the frames
    roll by along each round step, `(Rate+ <- Point-)[sites]`, and the squeeze of a step, half the strain's rate read
    back as a step, `(Point- <- Point-)[sites]`."""
    level = (time - first_time) / level_step
    early, share = int(level), level - int(level)
    # Each place's tetrahedron, read through the topology, and its corners' ramps there, which give the place's
    # barycentric weights.
    picked = cell_mesh.reading(cells, 3)                                        # numbers[sites <- cells]
    place_ramps = picked * ramps                                                # [corners] Plane+[sites]
    weights = (place_ramps & places) / (place_ramps.sum(axis=0) & places)       # [corners] Scalar+[sites]

    def blended(values):
        """A value stored at the vertices, `[levels, vertices]`, at the places and the time."""
        between = (values[early] * (1 - share) + values[early + 1] * share).field()   # [vertices]
        return (weights * (picked * (cell_mesh.incidence[3, 0] * between))).sum(axis=0)   # [sites]

    turn = picked * (level_turns[early] * (1 - share) + level_turns[early + 1] * share).field()   # (Rate+ <- Point-)[sites]
    # A round step, stretched by the strain, moves the origin; turned less twice the strain plus half its trace, it
    # turns the frame square to it; and the strain's change adds its turn.
    stretch = translating(blended(level_strains)(Step))                         # (Point- <- Point-)[sites]
    local = moving(Step + stretch / 2) + turning(Step - 2 * stretch + Step * (blended(level_traces) / 2)) + turn(Step)
    return local, translating(blended(level_strain_rates)) / 2

## Crystals in the waves

A gravitational wave is easy to misread from inside. The view sways, but that is the light, bent on its way to the eye. A free particle at rest is not moved at all: it keeps its place while the lengths between places change. Atoms bound to their neighbours are another matter: the wave stretches their bonds against their rest lengths, and the springs pull them into a motion of their own, measured against the free particles beside them, which no bending of light explains. Crystals of atoms on springs are honest strain gauges.

Little tetrahedra of atoms, bonded along their edges by springs and dashpots, sit side by side ahead of the eye. The squash drives a tetrahedron's swing at root two times the root of its springs' stiffness: the tuned crystal's stiffness puts that swing at the waves' frequency, the soft one's is a sixteenth of it and the stiff one's sixteen times it, all damped to a quarter of critical. An atom's velocity is a step of the origin in the frame at its place: it turns with the connection along the atom's motion, the squeeze slows it as space stretches, and the atom's bonds pull it along their proper separations, the steps the connection makes of their round steps. The soft crystal's bonds stretch with space, the stiff one's keep their length, and the tuned one's swing beyond the waves.

In [ ]:
# The crystals' stiffness against the tuned one's, their damping as a share of critical, and the time step of their
# atoms; the frames over one swing.
tunings = np.array([1 / 16, 1, 16])                                             # [crystals]
critical_share = 0.25
crystal_step = 0.01
n_frames = 75

In [ ]:
pick = core.picking(cell_mesh, cell_vertices)


def massive(bonds: Topology, rest: Scalar, spring_stiffness: Scalar, spring_damping: Scalar, starts: Point,
            start: float, step: float, times: np.ndarray) -> Iterator[tuple[Point, Scalar]]:
    """Atoms of unit mass rolled through the waves from rest at `starts`, `Point-[atoms]`, at time `start`, held by
    springs and dashpots along the edges of `bonds` of rest proper lengths `rest`, stiffness `spring_stiffness` and
    damping `spring_damping`, `Scalar+[bonds]` each, and stepped by `step` of time with symplectic Euler: their places
    and the bonds' proper strains, `Scalar+[bonds]`, at each of the `times`."""
    positions, lengths, time = starts.batch(), rest, start
    velocities = mv(Step, np.zeros((positions.shape[0], 3)))                    # [atoms] Point-
    tails = bonds.incidence[1, 0][0]                                            # numbers[bonds <- atoms]
    for target in times:
        while time < target - step / 2:
            places = positions.field()                                          # Point-[atoms]
            local, squeeze = connection(pick(positions), places, time)          # (Rate+ <- Point-)[atoms], (Point- <- Point-)[atoms]
            # Each bond's proper separation: the step the connection at its tail moves the origin by along its round
            # step.
            proper = origin.commutator((tails * local)(round_steps(bonds, places))).cast(Step)   # Point-[bonds]
            previous_lengths, lengths = lengths, proper.dot(proper).square_root()   # Scalar+[bonds], Scalar+[bonds]
            # Springs and dashpots along the proper separations, pulling a stretched bond's ends together.
            pulls = (spring_stiffness * (lengths - rest) + spring_damping * (lengths - previous_lengths) / step) / lengths   # Scalar+[bonds]
            forces = -(~bonds.d[0] * (pulls * proper)).batch()                  # [atoms] Point-
            # The connection and the squeeze at each atom as a batch, for the solve, which takes no field of maps.
            local, squeeze = local.batch(), squeeze.batch()                     # [atoms] Rate+ <- Point-, Point- <- Point-
            # The step each round step moves the origin by: a velocity's round step is the one it moves the origin by.
            stepping = origin.commutator(local(Step)).cast(Stretching)          # [atoms] Point- <- Point-
            # The velocity turned by the connection along the motion, squeezed and pushed by the springs; the turn's
            # step part only, cast so.
            carried_velocities = velocities - velocities.commutator(local(stepping.solve(velocities))).cast(Step) * step
            velocities = carried_velocities - squeeze(carried_velocities) * step + forces * step   # [atoms] Point-
            # Rolled along the round step of its velocity, on its great circle from the frame at its place.
            rolls = isoclinic(positions) * (moving(stepping.solve(velocities)) * (-step / 2)).exp()   # [atoms] Motor+
            positions = rolls >> origin                                         # [atoms] Point-
            time += step
        yield positions.field(), lengths / rest - 1

In [ ]:
# The crystals' tetrahedra of atoms, their bonds' stiffness and damping, and a bond at rest as long as its round step.
bonds, atoms = scenarios.crystals(scenarios.crystal_places())
stiffnesses = frequency ** 2 / 2 * tunings                                      # [crystals]
dampings = critical_share * np.sqrt(2 * stiffnesses)                            # [crystals]
n_bonds_each = bonds.n_elements[1] // len(tunings)
bond_stiffness = mv.scalar(np.repeat(stiffnesses, n_bonds_each)[:, None]).field()   # Scalar+[bonds]
bond_damping = mv.scalar(np.repeat(dampings, n_bonds_each)[:, None]).field()        # Scalar+[bonds]
bond_steps = round_steps(bonds, atoms)                                          # Point-[bonds]
bond_rest = bond_steps.dot(bond_steps).square_root()                            # Scalar+[bonds]
# The light from the crystals left them their distance before it reached the eye.
crystal_distance = scenarios.crystal_distance
eye_times = np.arange(n_frames) * period / n_frames                             # [frames]
states = massive(bonds, bond_rest, bond_stiffness, bond_damping, atoms,
                 eye_times[0] - crystal_distance - n_settling_swings * period, crystal_step, eye_times - crystal_distance)

## The view from inside

Every ray from the eye walks back through the tetrahedra in short legs, each straight across a tetrahedron's chart and read at its midpoint, its heading turned and squeezed by the same connection as the atoms, so the struts sway as the squash swings. Running back in time from the eye, a ray meets the waves as they were when its light passed. The last cell takes about half a minute, most of it rolling the atoms and tracing the frames.

In [ ]:
# A leg's greatest length, and a frame's duration in milliseconds.
leg_length = 0.02
duration_ms = 40

In [ ]:
# A ray's leg across a tetrahedron, in numga's dialect, from a place along a heading, a unit step of the origin in the
# frame there, having entered by a face, four where it starts inside, at the light's time there: the waves at a place
# blended over the tetrahedron's corners and the two nearest stored moments, as `connection` blends them.
walk = """
var<storage, read> ramps: array<Plane>;
var<storage, read> neighbours: array<u32>;
var<storage, read> entries: array<u32>;
var<storage, read> tet_vertices: array<u32>;
var<storage, read> strains: array<Strain>;
var<storage, read> strain_rates: array<Strain>;
var<storage, read> traces: array<f32>;
var<storage, read> turns: array<Turning>;
var<uniform> n_vertices: u32;
var<uniform> n_tets: u32;
var<uniform> first_time: f32;
var<uniform> level_step: f32;
var<uniform> origin: Point;
var<uniform> translating: Translating;
var<uniform> moving: Moving;
var<uniform> turning: Rolling;
var<uniform> leg_length: f32;

// The chart's plane, where the leg starts on it, the straight segment it runs, the arc it covers, the heading at its
// end, and the tetrahedron and face the next leg starts in.
struct Leg {
    plane: Plane,
    chart: Point,
    segment: Point,
    length: f32,
    heading: Step,
    tet: u32,
    entry: u32,
}

struct Local {
    strain: Strain,
    strain_rate: Strain,
    trace: f32,
    turn: Turning,
}

fn local_waves(tet: u32, place: Point, time: f32) -> Local {
    let level = (time - first_time) / level_step;
    let early = u32(floor(level));
    let share = level - floor(level);
    let now = early * n_vertices;
    let later = now + n_vertices;
    let unit = ramps[tet] + ramps[n_tets + tet] + ramps[2u * n_tets + tet] + ramps[3u * n_tets + tet];
    // A variable starts at nothing.
    var waves: Local;
    for (var corner = 0u; corner < 4u; corner++) {
        let vertex = tet_vertices[tet * 4u + corner];
        let weight = (ramps[corner * n_tets + tet] & place) / (unit & place);
        let early_weight = weight * (1.0 - share);
        let late_weight = weight * share;
        waves.strain = waves.strain + strains[now + vertex] * early_weight + strains[later + vertex] * late_weight;
        waves.strain_rate = waves.strain_rate + strain_rates[now + vertex] * early_weight
                            + strain_rates[later + vertex] * late_weight;
        waves.trace = waves.trace + traces[now + vertex] * early_weight + traces[later + vertex] * late_weight;
    }
    waves.turn = turns[early * n_tets + tet] * (1.0 - share) + turns[(early + 1u) * n_tets + tet] * share;
    return waves;
}

// The rate the frames roll by along a round step, as `connection` gives it.
fn rolling(steps: Step, waves: Local) -> Rate {
    let stretch = translating(waves.strain(steps));
    return moving(steps + stretch * 0.5) + turning(steps - stretch * 2.0 + steps * (waves.trace * 0.5))
           + waves.turn(steps);
}

// The round step a heading takes, less half the waves' stretch of it.
fn round_step(heading: Step, waves: Local) -> Step {
    return heading - translating(waves.strain(heading)) * 0.5;
}

// How a heading changes per length traced back in time: turned by the frames' roll along its round step, the turn
// kept a step of the origin, cast so, and stretched as the slices change, back in time, by the squeeze.
fn heading_change(heading: Step, waves: Local) -> Step {
    let reading = waves.strain_rate(heading);
    return -heading.commutator(rolling(round_step(heading, waves), waves)).cast(Step)
           + (translating(reading) - heading * (reading & heading)) * 0.5;
}

fn frame(place: Point) -> Motor {
    return ((1.0 + mv.xyzw) * (place / origin) + (1.0 - mv.xyzw)) * 0.5;
}

fn next_leg(place: Point, heading: Step, tet: u32, entry: u32, time: f32) -> Leg {
    let unit = ramps[tet] + ramps[n_tets + tet] + ramps[2u * n_tets + tet] + ramps[3u * n_tets + tet];
    let chart = place / (unit & place);
    let start_waves = local_waves(tet, place, time);
    // A leg's length along the round step at the start, and the share of it where the first corner's weight runs out.
    let step_end = (frame(place) * (moving(round_step(heading, start_waves)) * (-0.5 * leg_length)).exp()) >> origin;
    let step_segment = step_end / (unit & step_end) - chart;
    var share = 1.0;
    var exit = 4u;
    for (var corner = 0u; corner < 4u; corner++) {
        let change = ramps[corner * n_tets + tet] & step_segment;
        if (corner != entry && change < 0.0) {
            let run_out = max(-(ramps[corner * n_tets + tet] & chart) / change, 0.0);
            if (run_out < share) {
                share = run_out;
                exit = corner;
            }
        }
    }
    let length = acos(clamp(place.dot((chart + step_segment * share).normalized()), -1.0, 1.0));
    // The heading and the place half the leg on, and the waves there.
    let half_heading = (heading + heading_change(heading, start_waves) * (0.5 * length)).normalized();
    let half_place = (frame(place) * (moving(round_step(heading, start_waves)) * (-0.25 * length)).exp()) >> origin;
    let half_waves = local_waves(tet, half_place, time - 0.5 * length);
    // The leg rolled its length along the round step half way, from the frame at the start.
    let leg_end = (frame(place) * (moving(round_step(half_heading, half_waves)) * (-0.5 * length)).exp()) >> origin;
    var this_leg = Leg(unit, chart, leg_end / (unit & leg_end) - chart, length,
                       (heading + heading_change(half_heading, half_waves) * length).normalized(), tet, 4u);
    if (exit < 4u) {
        this_leg.tet = neighbours[tet * 4u + exit];
        this_leg.entry = entries[tet * 4u + exit];
    }
    return this_leg;
}
"""

In [ ]:
walk_values = dict(ramps=ramps, tet_vertices=cell_mesh.elements[3], strains=level_strains,
                   strain_rates=level_strain_rates, traces=level_traces, turns=level_turns,
                   n_vertices=cell_mesh.n_vertices, first_time=first_time, level_step=level_step, origin=origin,
                   translating=translating, moving=moving, turning=turning, leg_length=leg_length)
show_animation(render.waves_view(walk, walk_values, cell_mesh, cell_vertices, scenarios.eye, eye_times,
                                 scenarios.guardrails(cell_vertices, scenarios.n_guardrail_sides), bonds, atoms, states),
               "closed_universe_600_cell", duration_ms)

The 600-cell's twelve guardrail decagons, great circles side by side, are drawn each in its own colour, and every other edge as a thin ghost. Each crystal's bonds are coloured by their strain, blue squeezed and red stretched; from left to right the crystals are the soft, the tuned and the stiff one.

In [ ]:
# checks
# The round sphere's chords leave no deficit, and the squash is a fiftieth short of the continuum's eight.
assert (deficits(rest).abs() < 1e-12).all()
assert 7.5 < squash_squared < 8